# TinySLM — Grid A / Treino TS-01 no Kaggle

Compara 6 arquiteturas Llama pequenas sob a **mesma receita** e o mesmo prefixo de ~60M tokens.

**Fixos:** TS-01 limpo, BPE8K v01, contexto 512, EOS=2, packing contínuo, seed 42.
**Avaliação:** checkpoints próximos de 10M, 20M, 40M e 60M tokens.


In [ ]:
import os, json, math, time, random, zipfile, gc
from pathlib import Path

SEED=42
CONTEXT=512
VOCAB_SIZE=8000
TOKENS_PER_CANDIDATE=60_000_000
CHECKPOINT_TARGETS=[10_000_000,20_000_000,40_000_000,60_000_000]

MICRO_BATCH_SIZE=8
GRAD_ACCUM_STEPS=1
LEARNING_RATE=3e-4
WEIGHT_DECAY=0.1
WARMUP_RATIO=0.02
MAX_GRAD_NORM=1.0
EVAL_BATCHES=100
GEN_MAX_NEW_TOKENS=120

RUN_TRAINING=False

INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working/tinyslm_grid_a')
WORK_ROOT.mkdir(parents=True,exist_ok=True)

print('RUN_TRAINING:',RUN_TRAINING)
print('Orçamento/candidato:',f'{TOKENS_PER_CANDIDATE:,}')


## 1. Localizar o dataset `grid-tiny-v1`

In [ ]:
files=[p for p in INPUT_ROOT.rglob('*') if p.is_file()]
print('Ficheiros em /kaggle/input:',len(files))
for p in files[:60]: print(p)

def named(name, root=INPUT_ROOT):
    return [p for p in root.rglob(name) if p.is_file()]

tr=named('train.jsonl'); va=named('validation.jsonl')
if not (tr and va):
    zips=[p for p in files if p.suffix.lower()=='.zip']
    if not zips: raise FileNotFoundError('Não encontrei train.jsonl nem ZIP em /kaggle/input')
    ex=WORK_ROOT/'dataset'
    ex.mkdir(exist_ok=True)
    marker=ex/'.done'
    if not marker.exists():
        with zipfile.ZipFile(zips[0]) as z: z.extractall(ex)
        marker.write_text('ok')
    tr=named('train.jsonl',ex); va=named('validation.jsonl',ex)

TRAIN_FILE=tr[0]; VAL_FILE=va[0]
roots=[INPUT_ROOT,WORK_ROOT]
tokfiles=[]
for r in roots: tokfiles += list(r.rglob('tokenizer.json'))
if not tokfiles: raise FileNotFoundError('tokenizer.json não encontrado')
TOKENIZER_DIR=tokfiles[0].parent
mans=[]
for r in roots: mans += list(r.rglob('MANIFEST.json'))
MANIFEST_FILE=mans[0] if mans else None

print('TRAIN:',TRAIN_FILE)
print('VAL:',VAL_FILE)
print('TOKENIZER:',TOKENIZER_DIR)
print('MANIFEST:',MANIFEST_FILE)


## 2. Validar ambiente, tokenizer e corpus

In [ ]:
import torch, transformers, numpy as np, pandas as pd
from transformers import AutoTokenizer, LlamaConfig, LlamaForCausalLM

print('Torch:',torch.__version__)
print('Transformers:',transformers.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

tok=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True)
assert len(tok)==8000
assert tok.pad_token_id==0 and tok.bos_token_id==1 and tok.eos_token_id==2
print('Tokenizer OK: vocab=8000 PAD=0 BOS=1 EOS=2')

def count_lines(p):
    with p.open('rb') as f: return sum(1 for _ in f)
ntr=count_lines(TRAIN_FILE); nva=count_lines(VAL_FILE)
print('Train:',f'{ntr:,}','Validation:',f'{nva:,}')
assert ntr==1_799_239
assert nva==15_389
if MANIFEST_FILE:
    print('Manifest encontrado.')


## 3. Grid A

In [ ]:
GRID=[
 {'name':'A_h256_L6','hidden_size':256,'num_hidden_layers':6,'num_attention_heads':8,'intermediate_size':1024},
 {'name':'B_h256_L10','hidden_size':256,'num_hidden_layers':10,'num_attention_heads':8,'intermediate_size':1024},
 {'name':'C_h384_L6','hidden_size':384,'num_hidden_layers':6,'num_attention_heads':8,'intermediate_size':1536},
 {'name':'D_h384_L10','hidden_size':384,'num_hidden_layers':10,'num_attention_heads':8,'intermediate_size':1536},
 {'name':'E_h512_L8','hidden_size':512,'num_hidden_layers':8,'num_attention_heads':8,'intermediate_size':2048},
 {'name':'F_h512_L12','hidden_size':512,'num_hidden_layers':12,'num_attention_heads':8,'intermediate_size':2048},
]
def cfg(c):
    return LlamaConfig(vocab_size=VOCAB_SIZE,hidden_size=c['hidden_size'],
      intermediate_size=c['intermediate_size'],num_hidden_layers=c['num_hidden_layers'],
      num_attention_heads=c['num_attention_heads'],num_key_value_heads=c['num_attention_heads']//2,
      max_position_embeddings=CONTEXT,bos_token_id=1,eos_token_id=2,pad_token_id=0,
      tie_word_embeddings=True,attention_dropout=0.0,use_cache=False)

rows=[]
for c in GRID:
    m=LlamaForCausalLM(cfg(c)); n=sum(p.numel() for p in m.parameters())
    rows.append({**c,'parameters_M':round(n/1e6,3)})
    del m
gc.collect()
display(pd.DataFrame(rows))


## 4. Criar uma vez o mesmo prefixo packed de ~60M tokens para todos

In [ ]:
N_BLOCKS=math.ceil(TOKENS_PER_CANDIDATE/CONTEXT)
PACKED_TOKENS=N_BLOCKS*CONTEXT
PACK_FILE=WORK_ROOT/f'train_prefix_{PACKED_TOKENS}.uint16'

def build_prefix():
    if PACK_FILE.exists() and PACK_FILE.stat().st_size==PACKED_TOKENS*2:
        print('Prefixo existente:',PACK_FILE); return
    a=np.memmap(PACK_FILE,dtype=np.uint16,mode='w+',shape=(PACKED_TOKENS,))
    pos=0
    with TRAIN_FILE.open('r',encoding='utf-8') as f:
        for i,line in enumerate(f,1):
            text=json.loads(line)['text']
            ids=tok.encode(text,add_special_tokens=False)+[tok.eos_token_id]
            take=min(len(ids),PACKED_TOKENS-pos)
            if take:
                a[pos:pos+take]=np.asarray(ids[:take],dtype=np.uint16); pos+=take
            if i%50_000==0: print(f'{i:,} histórias | {pos:,}/{PACKED_TOKENS:,}')
            if pos>=PACKED_TOKENS: break
    a.flush()
    assert pos==PACKED_TOKENS
    print('Prefixo OK:',f'{PACKED_TOKENS:,}','tokens')
build_prefix()


## 5. Validation packed

In [ ]:
VAL_PACK=WORK_ROOT/'validation.uint16'
VAL_META=WORK_ROOT/'validation_meta.json'
if not (VAL_PACK.exists() and VAL_META.exists()):
    ids=[]
    with VAL_FILE.open('r',encoding='utf-8') as f:
        for line in f:
            text=json.loads(line)['text']
            ids.extend(tok.encode(text,add_special_tokens=False)); ids.append(tok.eos_token_id)
    usable=(len(ids)//CONTEXT)*CONTEXT
    np.asarray(ids[:usable],dtype=np.uint16).tofile(VAL_PACK)
    VAL_META.write_text(json.dumps({'raw':len(ids),'usable':usable,'blocks':usable//CONTEXT}))
meta=json.loads(VAL_META.read_text())
print(meta)


## 6. Datasets

In [ ]:
from torch.utils.data import Dataset,DataLoader
class Blocks(Dataset):
    def __init__(self,path,n):
        self.a=np.memmap(path,dtype=np.uint16,mode='r',shape=(n,))
        self.nb=n//CONTEXT
    def __len__(self): return self.nb
    def __getitem__(self,i):
        x=np.array(self.a[i*CONTEXT:(i+1)*CONTEXT],dtype=np.int64)
        t=torch.from_numpy(x)
        return {'input_ids':t,'labels':t.clone()}

train_ds=Blocks(PACK_FILE,PACKED_TOKENS)
val_ds=Blocks(VAL_PACK,meta['usable'])
print('Train blocks:',f'{len(train_ds):,}')
print('Val blocks:',f'{len(val_ds):,}')


## 7. Avaliação fixa

In [ ]:
PROMPTS=[
"Once upon a time, there was a little girl named Lily who",
"Tom was afraid of the dark, but one night",
"The dog could not find its ball, so",
"Anna opened the old box and was surprised because",
"Jack promised his mother that he would help her, but",
]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
AMP_DTYPE=torch.float16
from contextlib import nullcontext

@torch.no_grad()
def evaluate(model):
    model.eval(); losses=[]
    dl=DataLoader(val_ds,batch_size=MICRO_BATCH_SIZE,shuffle=False)
    for i,b in enumerate(dl):
        if i>=EVAL_BATCHES: break
        x=b['input_ids'].to(DEVICE); y=b['labels'].to(DEVICE)
        ctx=torch.autocast('cuda',dtype=AMP_DTYPE) if DEVICE=='cuda' else nullcontext()
        with ctx: o=model(input_ids=x,labels=y)
        losses.append(float(o.loss.cpu()))
    loss=sum(losses)/len(losses); model.train()
    return loss,math.exp(min(loss,20))

@torch.no_grad()
def samples(model):
    model.eval(); old=model.config.use_cache; model.config.use_cache=True
    out=[]
    for p in PROMPTS:
        x=tok(p,return_tensors='pt',add_special_tokens=False).to(DEVICE)
        y=model.generate(**x,max_new_tokens=120,do_sample=True,temperature=.8,top_p=.95,
                         eos_token_id=2,pad_token_id=0)
        out.append({'prompt':p,'text':tok.decode(y[0],skip_special_tokens=True)})
    model.config.use_cache=old; model.train()
    return out


## 8. Treino

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

TOKENS_MICRO=MICRO_BATCH_SIZE*CONTEXT
TOTAL_STEPS=math.ceil(PACKED_TOKENS/(TOKENS_MICRO*GRAD_ACCUM_STEPS))
WARMUP=max(1,int(TOTAL_STEPS*WARMUP_RATIO))

def lrfun(step):
    if step<WARMUP: return step/max(1,WARMUP)
    p=(step-WARMUP)/max(1,TOTAL_STEPS-WARMUP)
    return .5*(1+math.cos(math.pi*min(1,p)))

def train_one(c):
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    name=c['name']; rd=WORK_ROOT/'runs'/name; rd.mkdir(parents=True,exist_ok=True)
    model=LlamaForCausalLM(cfg(c)).to(DEVICE)
    params=sum(p.numel() for p in model.parameters())
    opt=AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    sch=LambdaLR(opt,lrfun)
    scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE=='cuda'))
    dl=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=False,pin_memory=True)
    seen=0; step=0; target_i=0; hist=[]; start=time.time()
    model.train(); opt.zero_grad(set_to_none=True)

    for mi,b in enumerate(dl,1):
        x=b['input_ids'].to(DEVICE,non_blocking=True); y=b['labels'].to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE,enabled=(DEVICE=='cuda')):
            o=model(input_ids=x,labels=y); loss=o.loss/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward(); seen+=x.numel()
        if mi%GRAD_ACCUM_STEPS==0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step(); step+=1

        if target_i<len(CHECKPOINT_TARGETS) and seen>=CHECKPOINT_TARGETS[target_i]:
            vl,ppl=evaluate(model)
            met={'candidate':name,'parameters':params,'target_tokens':CHECKPOINT_TARGETS[target_i],
                 'actual_tokens_seen':seen,'validation_loss':vl,'perplexity':ppl,
                 'train_loss':float(o.loss.detach().cpu()),'step':step,
                 'lr':sch.get_last_lr()[0],'minutes':(time.time()-start)/60,'samples':samples(model)}
            d=rd/f"{CHECKPOINT_TARGETS[target_i]//1_000_000:02d}M"; d.mkdir(exist_ok=True)
            model.save_pretrained(d,safe_serialization=True); tok.save_pretrained(d)
            (d/'metrics.json').write_text(json.dumps(met,indent=2,ensure_ascii=False))
            hist.append(met)
            print(f"[{name}] {seen/1e6:.2f}M | val={vl:.4f} | ppl={ppl:.2f} | {met['minutes']:.1f} min")
            target_i+=1
        if seen>=PACKED_TOKENS: break

    (rd/'history.json').write_text(json.dumps(hist,indent=2,ensure_ascii=False))
    del model,opt,sch,scaler; gc.collect(); torch.cuda.empty_cache()
    return hist


## 9. Gate — confirme antes de treinar

In [ ]:
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO')
print('Packed:',f'{PACKED_TOKENS:,}','tokens')
print('Candidatos:',[c['name'] for c in GRID])
print('RUN_TRAINING:',RUN_TRAINING)


## 10. Executar Grid A

Depois de validar tudo, volte à primeira célula de configuração e mude `RUN_TRAINING=True`.

In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available(): raise RuntimeError('GPU não disponível')
    for c in GRID:
        print('\n'+'='*70+'\n'+c['name']+'\n'+'='*70)
        train_one(c)
else:
    print('BLOQUEADO: validação concluída, treino não iniciado.')


## 11. Comparação

In [ ]:
records=[]
rr=WORK_ROOT/'runs'
if rr.exists():
    for p in rr.rglob('metrics.json'):
        m=json.loads(p.read_text()); records.append({k:v for k,v in m.items() if k!='samples'})
if records:
    df=pd.DataFrame(records).sort_values(['target_tokens','validation_loss'])
    display(df[['candidate','parameters','target_tokens','actual_tokens_seen','validation_loss',
                'perplexity','train_loss','minutes']])
    df.to_csv(WORK_ROOT/'grid_A_resultados.csv',index=False)
else:
    print('Sem resultados ainda.')


## 12. Saída

Tudo fica em `/kaggle/working/tinyslm_grid_a/`.

Após o treino, faça **Save Version** no Kaggle para preservar modelos, métricas e `grid_A_resultados.csv`.
